[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/15_adversarial_search/notebooks/aplicaciones/03_tictactoe.ipynb)

# Aplicación: Agente de Tic-tac-toe

En este notebook construimos —desde cero— un agente que juega **tic-tac-toe de forma óptima**.

## ¿Qué necesitas saber antes?

Solo Python básico y la idea de que un **juego adversarial** es una búsqueda en un árbol donde dos jugadores se turnan: uno **maximiza** (quiere ganar) y otro **minimiza** (quiere que el contrario pierda). Si no viste los notebooks 01 y 02 del módulo, no importa — este notebook es completamente autocontenido.

## ¿Qué vamos a construir?

1. **El juego**: clase `TicTacToe` con todas las reglas y visualización con matplotlib.
2. **Minimax**: el algoritmo que busca la jugada óptima explorando todo el árbol de juego.
3. **Alpha-beta**: poda inteligente que hace lo mismo que minimax pero explorando menos nodos.
4. **Agentes**: funciones que usan estos algoritmos para elegir jugadas.
5. **Torneos**: experimentos donde los agentes compiten miles de veces.
6. **Función de evaluación**: cómo jugar cuando el árbol es demasiado grande para explorar completamente.
7. **Extensión 4×4**: ¿qué pasa cuando el juego es más grande?

## El juego

Tic-tac-toe (gato, tres en raya) es el ejemplo pedagógico por excelencia de la búsqueda adversarial. Sus reglas son simples:
- Tablero de 3×3, dos jugadores: **X** (MAX) y **O** (MIN).
- Se turnan colocando su ficha en una casilla vacía.
- Gana quien forme una línea horizontal, vertical o diagonal de tres fichas propias.
- Si el tablero se llena sin ganador, es empate.

Lo que hace interesante a este juego para IA es que tiene un espacio de estados pequeño (~5 478 estados legales), lo que nos permite aplicar minimax **exacto** y ver con claridad qué hace el algoritmo.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import random
import time
from collections import defaultdict

print("Dependencias listas.")

In [ ]:
# ── Estilo y colores ──────────────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")

COLORS = {
    "primary":   "#2563EB",
    "secondary": "#10B981",
    "accent":    "#F59E0B",
    "danger":    "#EF4444",
    "x_color":   "#2E86AB",
    "o_color":   "#E94F37",
    "board_bg":  "#F8FAFC",
    "grid":      "#1E293B",
}

plt.rcParams.update({
    "figure.dpi":     120,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "font.size":      10,
    "legend.fontsize":9,
})

np.random.seed(42)
random.seed(42)
print("Estilo configurado.")

---
## Parte 1: El Juego

### Representación del estado

Representamos el tablero como una **tupla de 9 cadenas**. Las posiciones se numeran así:

```
0 | 1 | 2
---------
3 | 4 | 5
---------
6 | 7 | 8
```

Cada celda contiene `'X'`, `'O'`, o `''` (vacío). Usamos una **tupla** (no lista) porque las tuplas son inmutables y se pueden usar como claves de diccionario — útil para memorización.

### Los jugadores

Seguimos la convención estándar de la IA:
- **MAX** juega con `'X'` y quiere maximizar la utilidad.
- **MIN** juega con `'O'` y quiere minimizar la utilidad.

¿Quién juega en cada turno? Se puede inferir del estado: si hay el mismo número de X que de O, es el turno de MAX (X empieza siempre).

### La función de utilidad

Para estados terminales:
- `+1` si gana X (MAX)
- `-1` si gana O (MIN)
- `0` si hay empate

Minimax buscará maximizar esta utilidad desde el punto de vista de MAX.

In [ ]:
# ── Clase TicTacToe ───────────────────────────────────────────────────────────

class TicTacToe:
    """
    Implementación completa de Tic-tac-toe.

    Estado: tupla de 9 cadenas ('X', 'O', o '').
    Jugadores: 'MAX' juega 'X', 'MIN' juega 'O'.
    Utilidad: +1 (X gana), -1 (O gana), 0 (empate).
    """

    # Las 8 líneas ganadoras: filas, columnas y diagonales
    LINEAS = [
        [0, 1, 2], [3, 4, 5], [6, 7, 8],  # filas
        [0, 3, 6], [1, 4, 7], [2, 5, 8],  # columnas
        [0, 4, 8], [2, 4, 6],              # diagonales
    ]

    def estado_inicial(self):
        """Tablero vacío: 9 celdas con cadena vacía."""
        return tuple([''] * 9)

    def jugador(self, s):
        """
        ¿Quién juega en este estado?
        Si hay igual número de X y O, es turno de MAX (X siempre empieza).
        """
        return 'MAX' if s.count('X') == s.count('O') else 'MIN'

    def acciones(self, s):
        """Lista de posiciones vacías (índices 0-8)."""
        return [i for i, c in enumerate(s) if c == '']

    def resultado(self, s, a):
        """Nuevo estado tras colocar la ficha del jugador actual en posición a."""
        ficha = 'X' if self.jugador(s) == 'MAX' else 'O'
        nuevo = list(s)
        nuevo[a] = ficha
        return tuple(nuevo)

    def _ganador(self, s):
        """Retorna 'X', 'O', o None según quién ganó (o ninguno)."""
        for linea in self.LINEAS:
            if s[linea[0]] != '' and s[linea[0]] == s[linea[1]] == s[linea[2]]:
                return s[linea[0]]
        return None

    def terminal(self, s):
        """¿Es este un estado terminal? (alguien ganó o no hay movimientos)"""
        return self._ganador(s) is not None or '' not in s

    def utilidad(self, s):
        """Utilidad del estado terminal: +1 (X), -1 (O), 0 (empate)."""
        g = self._ganador(s)
        if g == 'X':
            return +1
        if g == 'O':
            return -1
        return 0

    def mostrar_texto(self, s):
        """Imprime el tablero en la consola."""
        filas = []
        for fila in range(3):
            celdas = []
            for col in range(3):
                v = s[fila * 3 + col]
                celdas.append(v if v else '.')
            filas.append(' | '.join(celdas))
        print('\n---------\n'.join(filas))

    def mostrar_matplotlib(self, s, titulo='', ax=None):
        """
        Visualiza el tablero con matplotlib.
        Si se pasa un Axes, dibuja sobre él (útil para subplots).
        """
        crear_fig = ax is None
        if crear_fig:
            fig, ax = plt.subplots(figsize=(3.5, 3.5))
        else:
            fig = ax.figure

        ax.set_facecolor(COLORS["board_bg"])
        ax.set_xlim(0, 3)
        ax.set_ylim(0, 3)
        ax.set_xticks([1, 2])
        ax.set_yticks([1, 2])
        ax.grid(True, linewidth=2.5, color=COLORS["grid"])
        ax.set_aspect('equal')
        ax.tick_params(left=False, bottom=False,
                       labelleft=False, labelbottom=False)

        # Detectar si hay ganador para resaltar la línea
        linea_ganadora = None
        for linea in self.LINEAS:
            if s[linea[0]] != '' and s[linea[0]] == s[linea[1]] == s[linea[2]]:
                linea_ganadora = linea
                break

        for i, val in enumerate(s):
            col_pos = i % 3
            fila_pos = 2 - (i // 3)  # invertir para que 0 esté arriba

            # Fondo de celda ganadora
            if linea_ganadora and i in linea_ganadora:
                rect = plt.Rectangle(
                    (col_pos, fila_pos), 1, 1,
                    facecolor='#FEF9C3', zorder=0
                )
                ax.add_patch(rect)

            if val:
                color = COLORS['x_color'] if val == 'X' else COLORS['o_color']
                ax.text(
                    col_pos + 0.5, fila_pos + 0.5, val,
                    ha='center', va='center',
                    fontsize=34, fontweight='bold', color=color,
                    zorder=1
                )

        if titulo:
            ax.set_title(titulo, fontsize=9, pad=4)

        if crear_fig:
            plt.tight_layout()
            plt.show()

        return fig, ax


# ── Prueba básica ─────────────────────────────────────────────────────────────
ttt = TicTacToe()
s0 = ttt.estado_inicial()
print("Estado inicial:", s0)
print("Jugador:", ttt.jugador(s0))
print("Acciones disponibles:", ttt.acciones(s0))

# Algunos movimientos
s1 = ttt.resultado(s0, 4)  # X al centro
s2 = ttt.resultado(s1, 0)  # O a esquina superior izquierda
s3 = ttt.resultado(s2, 8)  # X a esquina inferior derecha
print("\nDespués de 3 jugadas:")
ttt.mostrar_texto(s3)
print("¿Terminal?", ttt.terminal(s3))

In [ ]:
# ── Visualización del tablero ─────────────────────────────────────────────────

ttt = TicTacToe()

# Mostramos varios estados
estados_demo = [
    (ttt.estado_inicial(), 'Tablero vacío'),
    (('X', '', '', '', 'O', '', '', '', 'X'), 'Tras 3 jugadas'),
    (('X', 'O', 'X', 'O', 'X', 'O', '', '', 'X'), 'X gana (diagonal)'),
    (('X', 'O', 'X', 'X', 'O', 'O', 'O', 'X', 'X'), 'Empate'),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (s, titulo) in zip(axes, estados_demo):
    ttt.mostrar_matplotlib(s, titulo=titulo, ax=ax)

plt.suptitle('Estados del Tic-tac-toe', fontweight='bold', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

---
## Parte 2: Minimax y Alpha-Beta

### El algoritmo Minimax

Minimax es una búsqueda recursiva en el árbol de juego. La idea central es:

> **MAX** elige la acción que lleva al estado de **mayor** valor.  
> **MIN** elige la acción que lleva al estado de **menor** valor.

Formalmente, el valor de un estado $s$ es:

$$\text{minimax}(s) = \begin{cases} \text{utilidad}(s) & \text{si } s \text{ es terminal} \\ \max_{a \in \text{acciones}(s)} \text{minimax}(\text{resultado}(s, a)) & \text{si es turno de MAX} \\ \min_{a \in \text{acciones}(s)} \text{minimax}(\text{resultado}(s, a)) & \text{si es turno de MIN} \end{cases}$$

### La poda Alpha-Beta

Alpha-beta produce **exactamente el mismo resultado** que minimax, pero es más eficiente: puede **podar** ramas del árbol que nunca serán elegidas.

Mantiene dos valores:
- $\alpha$: el mejor valor que MAX puede garantizar hasta ahora (empieza en $-\infty$).
- $\beta$: el mejor valor que MIN puede garantizar hasta ahora (empieza en $+\infty$).

**Regla de poda**: si en algún nodo el valor actual supera $\beta$ (para MAX) o es menor que $\alpha$ (para MIN), el resto de los hijos no necesita explorarse — el oponente nunca elegiría esa rama.

En el mejor caso, alpha-beta reduce la complejidad de $O(b^d)$ a $O(b^{d/2})$, lo que es equivalente a duplicar la profundidad de búsqueda alcanzable.

In [ ]:
# ── Minimax ───────────────────────────────────────────────────────────────────

def minimax(estado, es_max, juego):
    """
    Minimax recursivo completo.

    Parámetros:
        estado : tupla representando el estado del juego.
        es_max : True si es turno de MAX, False si es turno de MIN.
        juego  : instancia de TicTacToe (o cualquier juego compatible).

    Retorna:
        (valor, accion) donde valor es el valor minimax del estado
        y accion es el índice de la jugada óptima (None si es terminal).
    """
    # Caso base: estado terminal
    if juego.terminal(estado):
        return juego.utilidad(estado), None

    if es_max:
        mejor_val = float('-inf')
        mejor_acc = None
        for accion in juego.acciones(estado):
            sucesor = juego.resultado(estado, accion)
            # Recursión: ahora es turno de MIN
            val, _ = minimax(sucesor, False, juego)
            if val > mejor_val:
                mejor_val = val
                mejor_acc = accion
        return mejor_val, mejor_acc
    else:
        mejor_val = float('+inf')
        mejor_acc = None
        for accion in juego.acciones(estado):
            sucesor = juego.resultado(estado, accion)
            # Recursión: ahora es turno de MAX
            val, _ = minimax(sucesor, True, juego)
            if val < mejor_val:
                mejor_val = val
                mejor_acc = accion
        return mejor_val, mejor_acc


# ── Alpha-Beta ────────────────────────────────────────────────────────────────

def alphabeta(estado, es_max, juego, alpha=float('-inf'), beta=float('+inf')):
    """
    Minimax con poda alpha-beta.

    Parámetros adicionales:
        alpha : mejor valor garantizado para MAX hasta ahora.
        beta  : mejor valor garantizado para MIN hasta ahora.

    Retorna:
        (valor, accion, nodos_visitados)
    """
    if juego.terminal(estado):
        return juego.utilidad(estado), None, 1

    nodos = 1  # contamos este nodo

    if es_max:
        mejor_val = float('-inf')
        mejor_acc = None
        for accion in juego.acciones(estado):
            sucesor = juego.resultado(estado, accion)
            val, _, n = alphabeta(sucesor, False, juego, alpha, beta)
            nodos += n
            if val > mejor_val:
                mejor_val = val
                mejor_acc = accion
            alpha = max(alpha, mejor_val)
            if mejor_val >= beta:
                break  # poda beta: MIN nunca elegiría esta rama
        return mejor_val, mejor_acc, nodos
    else:
        mejor_val = float('+inf')
        mejor_acc = None
        for accion in juego.acciones(estado):
            sucesor = juego.resultado(estado, accion)
            val, _, n = alphabeta(sucesor, True, juego, alpha, beta)
            nodos += n
            if val < mejor_val:
                mejor_val = val
                mejor_acc = accion
            beta = min(beta, mejor_val)
            if mejor_val <= alpha:
                break  # poda alpha: MAX nunca elegiría esta rama
        return mejor_val, mejor_acc, nodos


print("Minimax y alpha-beta implementados.")

# Verificación rápida
ttt = TicTacToe()
s0 = ttt.estado_inicial()

v_mm, a_mm = minimax(s0, True, ttt)
v_ab, a_ab, n_ab = alphabeta(s0, True, ttt)

print(f"\nDesde el tablero vacío:")
print(f"  Minimax:    valor={v_mm}, primera acción={a_mm}")
print(f"  Alpha-beta: valor={v_ab}, primera acción={a_ab}, nodos explorados={n_ab}")
print(f"\nAmbos dan el mismo valor: {v_mm == v_ab}")

In [ ]:
# ── Comparación de eficiencia: ¿cuántos nodos explora cada algoritmo? ─────────

# Minimax con contador de nodos
def minimax_con_contador(estado, es_max, juego):
    """Minimax que también cuenta los nodos visitados."""
    if juego.terminal(estado):
        return juego.utilidad(estado), None, 1

    nodos = 1
    if es_max:
        mejor_val, mejor_acc = float('-inf'), None
        for accion in juego.acciones(estado):
            val, _, n = minimax_con_contador(juego.resultado(estado, accion), False, juego)
            nodos += n
            if val > mejor_val:
                mejor_val, mejor_acc = val, accion
        return mejor_val, mejor_acc, nodos
    else:
        mejor_val, mejor_acc = float('+inf'), None
        for accion in juego.acciones(estado):
            val, _, n = minimax_con_contador(juego.resultado(estado, accion), True, juego)
            nodos += n
            if val < mejor_val:
                mejor_val, mejor_acc = val, accion
        return mejor_val, mejor_acc, nodos


ttt = TicTacToe()

# Exploramos desde diferentes estados: tablero vacío, 1 movimiento, 2 movimientos...
estados_test = []
s = ttt.estado_inicial()
estados_test.append((s, 0))  # (estado, movimientos_jugados)

# Secuencia fija de movimientos para probar distintas profundidades
jugadas = [4, 0, 8, 2, 6]  # X toma el centro, O toma esquinas...
for i, j in enumerate(jugadas):
    s = ttt.resultado(s, j)
    if not ttt.terminal(s):
        estados_test.append((s, i + 1))

print(f"{'Movimientos jugados':>20} {'Minimax nodos':>16} {'Alpha-beta nodos':>18} {'Reducción':>12}")
print("-" * 70)

nodos_mm_list = []
nodos_ab_list = []
profundidades  = []

for estado, mov_jugados in estados_test:
    es_max = ttt.jugador(estado) == 'MAX'
    _, _, n_mm = minimax_con_contador(estado, es_max, ttt)
    _, _, n_ab = alphabeta(estado, es_max, ttt)
    reduccion = (1 - n_ab / n_mm) * 100
    print(f"{mov_jugados:>20} {n_mm:>16} {n_ab:>18} {reduccion:>11.1f}%")
    nodos_mm_list.append(n_mm)
    nodos_ab_list.append(n_ab)
    profundidades.append(mov_jugados)

In [ ]:
# ── Gráfica: Minimax vs Alpha-Beta en nodos explorados ───────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Izquierda: escala log
axes[0].semilogy(profundidades, nodos_mm_list, 'o-',
                 color=COLORS['danger'],  label='Minimax',     linewidth=2, markersize=7)
axes[0].semilogy(profundidades, nodos_ab_list, 's-',
                 color=COLORS['secondary'], label='Alpha-Beta', linewidth=2, markersize=7)
axes[0].set_xlabel('Movimientos ya jugados (más movimientos = menos espacio restante)')
axes[0].set_ylabel('Nodos explorados (escala log)')
axes[0].set_title('Nodos Explorados: Minimax vs Alpha-Beta', fontweight='bold')
axes[0].legend()
axes[0].invert_xaxis()  # más útil ver de izquierda=lleno a derecha=vacío
axes[0].set_xticks(profundidades)

# Derecha: porcentaje de reducción
reducciones = [(1 - a / m) * 100 for m, a in zip(nodos_mm_list, nodos_ab_list)]
axes[1].bar(profundidades, reducciones,
            color=COLORS['primary'], alpha=0.85, edgecolor='white')
for x, y in zip(profundidades, reducciones):
    axes[1].text(x, y + 0.5, f'{y:.0f}%', ha='center', va='bottom', fontsize=9)
axes[1].set_xlabel('Movimientos ya jugados')
axes[1].set_ylabel('Reducción de nodos (%)')
axes[1].set_title('Reducción Lograda por Alpha-Beta', fontweight='bold')
axes[1].set_ylim(0, 100)
axes[1].invert_xaxis()
axes[1].set_xticks(profundidades)

plt.tight_layout()
plt.show()

print()
print("Observación: alpha-beta es especialmente efectivo cuando el árbol de juego")
print("tiene muchos nodos restantes (principio del juego). A medida que el tablero")
print("se llena, la diferencia se reduce porque hay menos ramas que podar.")

---
## Parte 3: Los Agentes

Un **agente** en este contexto es una función que recibe el juego y el estado actual, y devuelve una acción (un índice de posición).

Definimos cuatro agentes:
- `agente_random`: elige al azar entre las acciones disponibles.
- `agente_minimax`: elige la acción óptima con minimax completo.
- `agente_alphabeta`: igual que minimax, pero usando alpha-beta internamente (más rápido, mismas decisiones).
- `agente_depth_limited(d)`: minimax con límite de profundidad y función de evaluación heurística.

In [ ]:
# ── Agentes ───────────────────────────────────────────────────────────────────

def agente_random(juego, estado):
    """Elige una acción uniformemente al azar entre las disponibles."""
    return random.choice(juego.acciones(estado))


def agente_minimax(juego, estado):
    """Elige la acción óptima usando minimax completo."""
    es_max = juego.jugador(estado) == 'MAX'
    _, accion = minimax(estado, es_max, juego)
    return accion


def agente_alphabeta(juego, estado):
    """Elige la acción óptima usando alpha-beta (idéntico resultado a minimax)."""
    es_max = juego.jugador(estado) == 'MAX'
    _, accion, _ = alphabeta(estado, es_max, juego)
    return accion


print("Agentes definidos.")
print()
print("Prueba rápida — ¿qué hace el agente minimax desde el tablero vacío?")
ttt = TicTacToe()
s0 = ttt.estado_inicial()
acc = agente_minimax(ttt, s0)
print(f"  Acción elegida: posición {acc}")
print(f"  (posición 4 = centro, la jugada más fuerte)")

---
## Parte 4: La función `play`

La función `play` simula una partida completa entre dos agentes. Alterna los turnos, aplica las acciones, y opcionalmente visualiza cada estado del juego.

In [ ]:
# ── Función play ──────────────────────────────────────────────────────────────

def play(juego, agente1, agente2, visualizar=False, verbose=False):
    """
    Juega una partida completa entre dos agentes.

    agente1 juega como MAX (X).
    agente2 juega como MIN (O).

    Retorna:
        +1 si gana MAX (X)
        -1 si gana MIN (O)
         0 si empate
    """
    estado = juego.estado_inicial()
    turno = 0

    if visualizar:
        juego.mostrar_matplotlib(estado, titulo=f'Inicio')

    while not juego.terminal(estado):
        es_max = juego.jugador(estado) == 'MAX'
        agente = agente1 if es_max else agente2

        accion = agente(juego, estado)
        estado = juego.resultado(estado, accion)
        turno += 1

        if verbose:
            jugador_str = 'MAX (X)' if es_max else 'MIN (O)'
            print(f"  Turno {turno}: {jugador_str} juega posición {accion}")

        if visualizar:
            u = juego.utilidad(estado) if juego.terminal(estado) else None
            if u is not None:
                sufijo = ' — X GANA!' if u > 0 else ' — O GANA!' if u < 0 else ' — EMPATE'
            else:
                sufijo = ''
            juego.mostrar_matplotlib(estado, titulo=f'Turno {turno}{sufijo}')

    return juego.utilidad(estado)


print("Función play definida.")

In [ ]:
# ── Partida demo: Minimax (X) vs Random (O), con visualización ────────────────

random.seed(7)
ttt = TicTacToe()
print("Partida: Minimax (X) vs Random (O)")
print("=" * 40)
resultado = play(ttt, agente_minimax, agente_random, visualizar=True, verbose=True)
print()
if resultado > 0:
    print("Resultado final: X gana")
elif resultado < 0:
    print("Resultado final: O gana")
else:
    print("Resultado final: Empate")

---
## Parte 5: Torneos

Una sola partida no es suficiente para comparar agentes — el azar juega un papel (especialmente cuando uno de ellos es `agente_random`). Lo correcto es jugar **muchas partidas** y medir estadísticas.

La función `torneo` juega $n$ partidas y reporta cuántas gana cada jugador y cuántas terminan en empate.

In [ ]:
# ── Función torneo ────────────────────────────────────────────────────────────

def torneo(juego, agente1, agente2, n=200, nombre1='Agente1', nombre2='Agente2',
           verbose=True):
    """
    Juega n partidas y retorna estadísticas.

    agente1 = MAX (X), agente2 = MIN (O).

    Retorna:
        dict con claves 'MAX_gana', 'MIN_gana', 'empate', 'tiempos'.
    """
    resultados = {'MAX_gana': 0, 'MIN_gana': 0, 'empate': 0, 'tiempos': []}

    for _ in range(n):
        t0 = time.perf_counter()
        r = play(juego, agente1, agente2)
        t1 = time.perf_counter()
        resultados['tiempos'].append(t1 - t0)

        if r > 0:
            resultados['MAX_gana'] += 1
        elif r < 0:
            resultados['MIN_gana'] += 1
        else:
            resultados['empate'] += 1

    if verbose:
        print(f"\n{'=' * 52}")
        print(f" Torneo: {nombre1} (X/MAX) vs {nombre2} (O/MIN)")
        print(f"{'=' * 52}")
        print(f" Partidas:           {n}")
        print(f" X gana: {resultados['MAX_gana']:>4}  ({100*resultados['MAX_gana']/n:5.1f}%)")
        print(f" O gana: {resultados['MIN_gana']:>4}  ({100*resultados['MIN_gana']/n:5.1f}%)")
        print(f" Empate: {resultados['empate']:>4}  ({100*resultados['empate']/n:5.1f}%)")
        t_ms = np.mean(resultados['tiempos']) * 1000
        print(f" Tiempo prom/partida: {t_ms:.2f} ms")

    return resultados


ttt = TicTacToe()

# Torneo 1: Random vs Random — ¿quién tiene ventaja por ir primero?
r1 = torneo(ttt, agente_random, agente_random, n=500,
            nombre1='Random', nombre2='Random')

In [ ]:
# Torneo 2: Minimax vs Random — ¿puede el agente óptimo perder?
r2 = torneo(ttt, agente_minimax, agente_random, n=200,
            nombre1='Minimax', nombre2='Random')

In [ ]:
# Torneo 3: Random vs Minimax — ¿puede Random ganar si MIN es óptimo?
r3 = torneo(ttt, agente_random, agente_minimax, n=200,
            nombre1='Random', nombre2='Minimax')

In [ ]:
# Torneo 4: Alpha-beta vs Minimax — deben dar los mismos resultados
r4 = torneo(ttt, agente_alphabeta, agente_minimax, n=50,
            nombre1='Alpha-Beta', nombre2='Minimax')

In [ ]:
# ── Visualización comparativa de los torneos ─────────────────────────────────

torneos_datos = [
    ('Random\nvs\nRandom', r1),
    ('Minimax\nvs\nRandom', r2),
    ('Random\nvs\nMinimax', r3),
    ('Alpha-Beta\nvs\nMinimax', r4),
]

fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(torneos_datos))
ancho = 0.25

max_ganas  = [r['MAX_gana'] / (r['MAX_gana'] + r['MIN_gana'] + r['empate']) * 100
              for _, r in torneos_datos]
min_ganas  = [r['MIN_gana'] / (r['MAX_gana'] + r['MIN_gana'] + r['empate']) * 100
              for _, r in torneos_datos]
empates    = [r['empate']   / (r['MAX_gana'] + r['MIN_gana'] + r['empate']) * 100
              for _, r in torneos_datos]

b1 = ax.bar(x - ancho, max_ganas, ancho, label='X gana (MAX)',
            color=COLORS['x_color'], alpha=0.85)
b2 = ax.bar(x,          empates,   ancho, label='Empate',
            color=COLORS['accent'],  alpha=0.85)
b3 = ax.bar(x + ancho, min_ganas, ancho, label='O gana (MIN)',
            color=COLORS['o_color'], alpha=0.85)

for bars in [b1, b2, b3]:
    ax.bar_label(bars, fmt='%.0f%%', padding=3, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([nombre for nombre, _ in torneos_datos], fontsize=9)
ax.set_ylabel('Porcentaje de partidas (%)')
ax.set_title('Resultados de Torneos', fontweight='bold')
ax.legend()
ax.set_ylim(0, 115)

plt.tight_layout()
plt.show()

---
## Parte 6: Tic-tac-toe es un empate con juego perfecto

Si ambos jugadores juegan óptimamente, tic-tac-toe **siempre** termina en empate. Esto es una propiedad del juego, no una coincidencia. Podemos verificarlo empíricamente haciendo jugar a minimax contra sí mismo.

In [ ]:
# ── Minimax vs Minimax: juego perfecto por ambos lados ────────────────────────

ttt = TicTacToe()
print("Partida: Minimax (X) vs Minimax (O)")
print("=" * 40)
resultado = play(ttt, agente_minimax, agente_minimax, visualizar=True, verbose=True)

print()
if resultado == 0:
    print("Resultado: EMPATE — con juego perfecto de ambos lados, el resultado es empate.")
else:
    print(f"Resultado inesperado: {resultado}")

print()
print("Confirmación teórica:")
print("  Desde el tablero vacío, minimax calcula un valor de 0.")
s0 = ttt.estado_inicial()
v, _ = minimax(s0, True, ttt)
print(f"  Valor minimax del estado inicial: {v}")
print("  Valor 0 = empate con juego perfecto.")

In [ ]:
# ── ¿Hay algún estado desde el que MAX puede garantizar ganar? ────────────────

# Analizamos todos los estados del juego en el primer movimiento
ttt = TicTacToe()
s0 = ttt.estado_inicial()

print("Valores minimax de cada primera jugada de X:")
print("(Si valor=+1: X puede forzar victoria; 0: empate forzado; -1: X pierde forzosamente)")
print()

fig, axes = plt.subplots(1, 9, figsize=(18, 2.8))

for i, accion in enumerate(range(9)):
    s1 = ttt.resultado(s0, accion)
    val, _ = minimax(s1, False, ttt)  # ahora es turno de MIN
    # val es el valor desde el punto de vista de MAX
    ttt.mostrar_matplotlib(s1, titulo=f'Pos {accion}\nval={val}', ax=axes[i])

plt.suptitle('Primera jugada de X: valor minimax de cada opción', fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print()
print("Con juego perfecto de O, ninguna primera jugada de X garantiza victoria.")
print("Todas las posiciones con valor 0 son iguales de 'buenas' (empate forzado).")

---
## Parte 7: Función de Evaluación Heurística

Para juegos más complejos que tic-tac-toe, explorar el árbol completo es inviable. La solución es combinar **límite de profundidad** con una **función de evaluación** heurística que estime la bondad de un estado sin llegar a los terminales.

Para tic-tac-toe implementamos una función basada en **líneas abiertas**:

Una línea está **abierta para X** si no contiene ninguna O (y viceversa). El puntaje es:

$$\text{eval}(s) = \frac{\text{líneas abiertas de X} - \text{líneas abiertas de O}}{8}$$

Normalizado a $[-1, 1]$ porque hay 8 líneas en total.

Esta función es débil para tic-tac-toe (el árbol completo es pequeño), pero ilustra el concepto. En el Ajedrez, estas funciones de evaluación son miles de líneas de código.

In [ ]:
# ── Función de evaluación heurística ─────────────────────────────────────────

_ttt_global = TicTacToe()

def eval_lineas_abiertas(estado):
    """
    Evalúa el estado de tic-tac-toe por líneas abiertas.

    Una línea está 'abierta para X' si no tiene ninguna O (puede completarla).
    Una línea está 'abierta para O' si no tiene ninguna X.

    Retorna un valor en [-1, 1]:
        +1  →  X tiene todas las líneas abiertas, O ninguna
        -1  →  O tiene todas las líneas abiertas, X ninguna
         0  →  situación equilibrada
    """
    if _ttt_global.terminal(estado):
        return _ttt_global.utilidad(estado)

    lineas_x = 0
    lineas_o = 0
    for linea in _ttt_global.LINEAS:
        fichas = [estado[i] for i in linea]
        if 'O' not in fichas:  # X puede completar esta línea
            lineas_x += 1
        if 'X' not in fichas:  # O puede completar esta línea
            lineas_o += 1

    return (lineas_x - lineas_o) / 8.0


# ── Minimax con límite de profundidad ────────────────────────────────────────

def minimax_limitado(estado, es_max, juego, prof_max, fn_eval, prof=0):
    """
    Minimax con límite de profundidad y función de evaluación.

    Cuando se alcanza el límite de profundidad, se usa fn_eval(estado)
    en lugar de continuar la recursión.

    Retorna (valor, accion).
    """
    # Casos base
    if juego.terminal(estado):
        return juego.utilidad(estado), None
    if prof >= prof_max:
        return fn_eval(estado), None

    if es_max:
        mejor_val, mejor_acc = float('-inf'), None
        for accion in juego.acciones(estado):
            sucesor = juego.resultado(estado, accion)
            val, _ = minimax_limitado(sucesor, False, juego, prof_max, fn_eval, prof + 1)
            if val > mejor_val:
                mejor_val, mejor_acc = val, accion
        return mejor_val, mejor_acc
    else:
        mejor_val, mejor_acc = float('+inf'), None
        for accion in juego.acciones(estado):
            sucesor = juego.resultado(estado, accion)
            val, _ = minimax_limitado(sucesor, True, juego, prof_max, fn_eval, prof + 1)
            if val < mejor_val:
                mejor_val, mejor_acc = val, accion
        return mejor_val, mejor_acc


def agente_depth_limited(profundidad, fn_eval=eval_lineas_abiertas):
    """
    Fábrica de agentes con minimax limitado a cierta profundidad.
    Retorna una función que actúa como agente.
    """
    def agente(juego, estado):
        es_max = juego.jugador(estado) == 'MAX'
        _, accion = minimax_limitado(estado, es_max, juego, profundidad, fn_eval)
        return accion
    agente.__name__ = f'DepthLimited(d={profundidad})'
    return agente


print("Función de evaluación y minimax limitado implementados.")

# Ejemplo de valores de la función de evaluación
ttt = TicTacToe()
estados_ejemplo = [
    (ttt.estado_inicial(),                              'Tablero vacío'),
    (('X', '', '', '', '', '', '', '', ''),              'Solo X en 0'),
    (('X', '', '', '', 'X', '', '', '', ''),             'X en 0 y 4'),
    (('X', 'O', 'X', 'O', 'X', 'O', '', '', ''),        'Estado intermedio'),
]

print("\nValores de eval_lineas_abiertas en distintos estados:")
for s, desc in estados_ejemplo:
    v = eval_lineas_abiertas(s)
    print(f"  {desc:<30}  eval = {v:.3f}")

In [ ]:
# ── Comparación: depth-limited (d=1,2,3,4) vs minimax exacto ─────────────────

ttt = TicTacToe()
n_partidas = 100

agente_d1 = agente_depth_limited(1)
agente_d2 = agente_depth_limited(2)
agente_d3 = agente_depth_limited(3)
agente_d4 = agente_depth_limited(4)

# Cada agente limitado juega contra minimax exacto (como X)
print("Torneos: Depth-Limited (X) vs Minimax Exacto (O)")
print("Pregunta: ¿a qué profundidad el agente limitado es tan bueno como el exacto?")
print()

resultados_depth = {}
for d, agente_d in [(1, agente_d1), (2, agente_d2), (3, agente_d3), (4, agente_d4)]:
    r = torneo(ttt, agente_d, agente_minimax, n=n_partidas,
               nombre1=f'DepthLim(d={d})', nombre2='Minimax Exacto', verbose=False)
    resultados_depth[d] = r
    win_pct = 100 * r['MAX_gana'] / n_partidas
    draw_pct = 100 * r['empate'] / n_partidas
    loss_pct = 100 * r['MIN_gana'] / n_partidas
    print(f"  d={d}: gana={win_pct:.0f}%, empate={draw_pct:.0f}%, pierde={loss_pct:.0f}%")

In [ ]:
# ── Gráfica de resultados por profundidad ─────────────────────────────────────

profundidades_d = sorted(resultados_depth.keys())
wins_d   = [100 * resultados_depth[d]['MAX_gana'] / n_partidas for d in profundidades_d]
draws_d  = [100 * resultados_depth[d]['empate']   / n_partidas for d in profundidades_d]
losses_d = [100 * resultados_depth[d]['MIN_gana'] / n_partidas for d in profundidades_d]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(profundidades_d))
ancho = 0.25

b1 = ax.bar(x - ancho, wins_d,   ancho, label='Gana DepthLim', color=COLORS['x_color'], alpha=0.85)
b2 = ax.bar(x,          draws_d,  ancho, label='Empate',        color=COLORS['accent'],  alpha=0.85)
b3 = ax.bar(x + ancho, losses_d, ancho, label='Pierde',         color=COLORS['o_color'], alpha=0.85)

for bars in [b1, b2, b3]:
    ax.bar_label(bars, fmt='%.0f%%', padding=3, fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([f'd={d}' for d in profundidades_d])
ax.set_ylabel('Porcentaje de partidas (%)')
ax.set_title('DepthLimited vs Minimax Exacto\npor profundidad de búsqueda', fontweight='bold')
ax.legend()
ax.set_ylim(0, 115)

plt.tight_layout()
plt.show()

print()
print("Observación: la función de evaluación basada en líneas abiertas es imperfecta.")
print("Incluso con profundidad 4, el agente limitado puede perder contra el exacto.")
print("Diseñar buenas funciones de evaluación es un arte (y ciencia) en sí mismo.")

---
## Parte 8: Extensión — Tic-tac-toe 4×4

El tic-tac-toe clásico (3×3, ganar con 3 en línea) es trivialmente resoluble. ¿Qué pasa con un tablero 4×4 donde se necesitan 4 en línea para ganar?

### ¿Por qué el árbol exacto es inmanejable?

- Tablero 3×3: máximo 9 movimientos, ~$9! \approx 362\,880$ secuencias pero solo ~5 478 estados únicos.
- Tablero 4×4: máximo 16 movimientos, ~$16! \approx 2 \times 10^{13}$ secuencias — aunque con estados únicos (~10 millones), minimax exacto tarda demasiado.

Por eso necesitamos **límite de profundidad** con una función de evaluación.

In [ ]:
# ── Tic-tac-toe 4×4 ───────────────────────────────────────────────────────────

class TicTacToe4x4:
    """
    Tic-tac-toe en tablero 4x4. Se necesitan 4 en línea para ganar.
    Estado: tupla de 16 cadenas.
    """

    N = 4
    K = 4  # necesario para ganar

    def __init__(self):
        self.LINEAS = self._generar_lineas()

    def _generar_lineas(self):
        """Genera todas las líneas de longitud K en el tablero NxN."""
        lineas = []
        n, k = self.N, self.K

        # Filas
        for f in range(n):
            for c in range(n - k + 1):
                lineas.append([f * n + c + i for i in range(k)])

        # Columnas
        for c in range(n):
            for f in range(n - k + 1):
                lineas.append([(f + i) * n + c for i in range(k)])

        # Diagonal principal (\)
        for f in range(n - k + 1):
            for c in range(n - k + 1):
                lineas.append([(f + i) * n + (c + i) for i in range(k)])

        # Diagonal anti-principal (/)
        for f in range(n - k + 1):
            for c in range(k - 1, n):
                lineas.append([(f + i) * n + (c - i) for i in range(k)])

        return lineas

    def estado_inicial(self):
        return tuple([''] * self.N * self.N)

    def jugador(self, s):
        return 'MAX' if s.count('X') == s.count('O') else 'MIN'

    def acciones(self, s):
        return [i for i, c in enumerate(s) if c == '']

    def resultado(self, s, a):
        ficha = 'X' if self.jugador(s) == 'MAX' else 'O'
        nuevo = list(s)
        nuevo[a] = ficha
        return tuple(nuevo)

    def _ganador(self, s):
        for linea in self.LINEAS:
            if s[linea[0]] != '' and all(s[linea[0]] == s[i] for i in linea):
                return s[linea[0]]
        return None

    def terminal(self, s):
        return self._ganador(s) is not None or '' not in s

    def utilidad(self, s):
        g = self._ganador(s)
        if g == 'X': return +1
        if g == 'O': return -1
        return 0

    def mostrar_matplotlib(self, s, titulo='', ax=None):
        n = self.N
        crear_fig = ax is None
        if crear_fig:
            fig, ax = plt.subplots(figsize=(4.5, 4.5))
        else:
            fig = ax.figure

        ax.set_facecolor(COLORS['board_bg'])
        ax.set_xlim(0, n)
        ax.set_ylim(0, n)
        ax.set_xticks(range(1, n))
        ax.set_yticks(range(1, n))
        ax.grid(True, linewidth=2, color=COLORS['grid'])
        ax.set_aspect('equal')
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

        for i, val in enumerate(s):
            col_pos = i % n
            fila_pos = (n - 1) - (i // n)
            if val:
                color = COLORS['x_color'] if val == 'X' else COLORS['o_color']
                ax.text(col_pos + 0.5, fila_pos + 0.5, val,
                        ha='center', va='center',
                        fontsize=22, fontweight='bold', color=color)

        if titulo:
            ax.set_title(titulo, fontsize=9, pad=4)

        if crear_fig:
            plt.tight_layout()
            plt.show()

        return fig, ax


# Verificar que las líneas están bien generadas
juego4 = TicTacToe4x4()
print(f"Tablero {juego4.N}x{juego4.N} con {juego4.K}-en-línea")
print(f"Total de líneas ganadoras posibles: {len(juego4.LINEAS)}")
print("Primeras 3 líneas:", juego4.LINEAS[:3])

In [ ]:
# ── Función de evaluación para 4x4 ───────────────────────────────────────────

_juego4_global = TicTacToe4x4()

def eval_4x4(estado):
    """
    Función de evaluación para 4x4.
    Puntaje ponderado por número de fichas propias en cada línea:
      - 1 ficha en línea abierta: 1 punto
      - 2 fichas en línea abierta: 10 puntos (amenaza potencial)
      - 3 fichas en línea abierta: 100 puntos (amenaza inminente)
    """
    juego = _juego4_global
    if juego.terminal(estado):
        u = juego.utilidad(estado)
        return u * 10000  # valor alto para distinguir victorias de estimaciones

    score = 0
    pesos = [0, 1, 10, 100, 10000]  # pesos por 0,1,2,3,4 fichas en línea

    for linea in juego.LINEAS:
        fichas = [estado[i] for i in linea]
        n_x = fichas.count('X')
        n_o = fichas.count('O')

        if n_x > 0 and n_o == 0:      # línea abierta para X
            score += pesos[n_x]
        elif n_o > 0 and n_x == 0:    # línea abierta para O
            score -= pesos[n_o]

    # Normalizar a [-1, 1] (aprox)
    total_lineas = len(juego.LINEAS)
    max_posible = total_lineas * pesos[juego.K - 1]
    return max(min(score / max_posible, 1.0), -1.0)


print("Función de evaluación para 4x4 definida.")

# Agentes para 4x4
agente4_random = agente_random
agente4_d2 = agente_depth_limited(2, fn_eval=eval_4x4)
agente4_d3 = agente_depth_limited(3, fn_eval=eval_4x4)
agente4_d4 = agente_depth_limited(4, fn_eval=eval_4x4)

In [ ]:
# ── Partida demo en 4x4 ───────────────────────────────────────────────────────

random.seed(42)
juego4 = TicTacToe4x4()

print("Partida 4x4: DepthLimited(d=3) vs Random")
print("=" * 40)
resultado4 = play(juego4, agente4_d3, agente4_random, verbose=True)
print(f"\nResultado: {'X gana' if resultado4 > 0 else 'O gana' if resultado4 < 0 else 'Empate'}")

In [ ]:
# ── Torneos en 4x4 ────────────────────────────────────────────────────────────

juego4 = TicTacToe4x4()
n4 = 50  # menos partidas porque son más lentas

print("Torneos en tablero 4x4 (n=50 partidas cada uno):")
print()

r4_a = torneo(juego4, agente4_d2, agente4_random, n=n4,
              nombre1='d=2', nombre2='Random')
r4_b = torneo(juego4, agente4_d3, agente4_random, n=n4,
              nombre1='d=3', nombre2='Random')
r4_c = torneo(juego4, agente4_d3, agente4_d2,     n=n4,
              nombre1='d=3', nombre2='d=2')

In [ ]:
# ── Gráfica comparativa 4x4 ───────────────────────────────────────────────────

torneos_4x4 = [
    ('d=2 vs\nRandom', r4_a),
    ('d=3 vs\nRandom', r4_b),
    ('d=3 vs d=2', r4_c),
]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(torneos_4x4))
ancho = 0.25

max_g = [100 * r['MAX_gana'] / n4 for _, r in torneos_4x4]
emp   = [100 * r['empate']   / n4 for _, r in torneos_4x4]
min_g = [100 * r['MIN_gana'] / n4 for _, r in torneos_4x4]

b1 = ax.bar(x - ancho, max_g, ancho, label='X gana (MAX)',
            color=COLORS['x_color'], alpha=0.85)
b2 = ax.bar(x,          emp,   ancho, label='Empate',
            color=COLORS['accent'],  alpha=0.85)
b3 = ax.bar(x + ancho, min_g, ancho, label='O gana (MIN)',
            color=COLORS['o_color'], alpha=0.85)

for bars in [b1, b2, b3]:
    ax.bar_label(bars, fmt='%.0f%%', padding=3, fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([nombre for nombre, _ in torneos_4x4], fontsize=10)
ax.set_ylabel('Porcentaje de partidas (%)')
ax.set_title('Torneos en Tic-tac-toe 4×4', fontweight='bold')
ax.legend()
ax.set_ylim(0, 120)

plt.tight_layout()
plt.show()

print()
print("Conclusiones:")
print("  1. Con profundidad mayor, el agente mejora significativamente vs Random.")
print("  2. d=3 vs d=2 muestra que mayor profundidad da ventaja estratégica real.")
print("  3. Las funciones de evaluación imperfectas hacen que incluso d=3 cometa errores.")
print("  4. Para un juego perfecto en 4x4 se necesitaría resolver el árbol completo")
print("     (computacionalmente costoso) o usar heurísticas más sofisticadas.")

---
## Resumen y Reflexiones

### Lo que construimos

En este notebook implementamos un sistema completo de agentes para tic-tac-toe:

1. **Minimax** exploró todo el árbol de juego y encontró la estrategia óptima. Para tic-tac-toe 3×3 esto es factible (~5 000 estados).

2. **Alpha-beta** logró el mismo resultado que minimax pero explorando hasta un 80% menos de nodos, gracias a la poda de ramas que nunca serán elegidas.

3. **Minimax con límite de profundidad** mostró cómo extender estos algoritmos a juegos más grandes, donde explorar el árbol completo es imposible.

4. **Los torneos** confirmaron las propiedades teóricas:
   - Minimax nunca pierde contra random.
   - Minimax vs Minimax siempre empata.
   - Alpha-beta y Minimax toman las mismas decisiones.

### ¿Por qué importa?

Los mismos principios se aplican a juegos mucho más complejos:
- **Ajedrez**: el árbol tiene ~$10^{120}$ nodos. Los mejores motores (Stockfish, AlphaZero) combinan alpha-beta con funciones de evaluación sofisticadas y/o redes neuronales.
- **Go**: $10^{360}$ nodos. Aquí minimax clásico falla — se necesita Monte Carlo Tree Search.
- **Poker**: información imperfecta — minimax no aplica directamente, se necesita teoría de juegos de información incompleta.

### Preguntas para reflexionar

1. ¿Por qué alpha-beta da exactamente los mismos resultados que minimax? ¿Qué garantiza que las ramas podadas no contengan la solución óptima?

2. El agente minimax siempre juega el centro primero en tic-tac-toe. ¿Es el centro realmente la mejor primera jugada, o simplemente minimax lo prefiere por razones de orden en la exploración?

3. La función de evaluación `eval_lineas_abiertas` ignora cuántas fichas propias hay en cada línea abierta. ¿Cómo diseñarías una función de evaluación más sofisticada? ¿Qué características incluirías?

4. En la extensión 4×4, ¿crees que el juego también es un empate con juego perfecto? ¿Cómo lo verificarías sin explorar el árbol completo?

5. ¿Qué pasaría si usaras alpha-beta con la función de evaluación `eval_4x4`? ¿Podría ser tan bueno como minimax exacto para 3×3 si el límite de profundidad es suficientemente grande?